# Week 3 – PySpark Transaction Volume Analysis
**Capstone: Personal Expense Monitoring System**

## Setup

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, avg, count, date_format

spark = SparkSession.builder.appName("ExpenseMonitor").getOrCreate()
print("Spark session started")

Spark session started


## Load Dataset

In [2]:
# Upload your expenses.csv file
from google.colab import files

uploaded = files.upload()

Saving transactions.csv to transactions.csv


In [3]:
df = spark.read.csv("transactions.csv", header=True, inferSchema=True)
df = df.withColumn("month", date_format(col("transaction_date"), "yyyy-MM"))
print("Total records:", df.count())
df.show()

Total records: 28
+--------------+-------+----------+-------------+------+----------------+-------+
|transaction_id|user_id| user_name|     category|amount|transaction_date|  month|
+--------------+-------+----------+-------------+------+----------------+-------+
|             1|      1|Priya Nair|         Food|   180|      2025-01-03|2025-01|
|             2|      1|Priya Nair|    Transport|    60|      2025-01-07|2025-01|
|             3|      1|Priya Nair|    Utilities|   140|      2025-01-12|2025-01|
|             4|      1|Priya Nair|       Health|    90|      2025-01-18|2025-01|
|             5|      1|Priya Nair|         Food|   220|      2025-01-25|2025-01|
|             6|      1|Priya Nair|Entertainment|    50|      2025-01-28|2025-01|
|             7|      2|Ravi Kumar|         Food|   130|      2025-01-05|2025-01|
|             8|      2|Ravi Kumar|    Transport|    75|      2025-01-10|2025-01|
|             9|      2|Ravi Kumar|       Health|   800|      2025-01-15|2025-01

## Monthly Total Spend Per User

In [4]:
monthly_spend = df.groupBy("user_id", "user_name", "month")     .agg(sum("amount").alias("monthly_total"), count("amount").alias("num_transactions"))

print("Monthly Total Spend Per User")
monthly_spend.orderBy("user_name", "month").show()

Monthly Total Spend Per User
+-------+----------+-------+-------------+----------------+
|user_id| user_name|  month|monthly_total|num_transactions|
+-------+----------+-------+-------------+----------------+
|      3| Meena Das|2025-01|         1795|               5|
|      3| Meena Das|2025-02|          575|               4|
|      1|Priya Nair|2025-01|          740|               6|
|      1|Priya Nair|2025-02|          495|               4|
|      2|Ravi Kumar|2025-01|         1155|               5|
|      2|Ravi Kumar|2025-02|          430|               4|
+-------+----------+-------+-------------+----------------+



## Average Transaction Amount

In [5]:
avg_amount = df.agg(avg("amount").alias("avg_amount")).collect()[0]["avg_amount"]
print(f"Overall Average Transaction: ₹{avg_amount:.2f}")

Overall Average Transaction: ₹185.36


## Unusual Spending Detection (> 2x Average)

In [6]:
unusual = df.filter(col("amount") > avg_amount * 2)
print("Unusual / Large One-Time Transactions")
unusual.select("user_name", "category", "amount", "transaction_date").show()

Unusual / Large One-Time Transactions
+----------+--------+------+----------------+
| user_name|category|amount|transaction_date|
+----------+--------+------+----------------+
|Ravi Kumar|  Health|   800|      2025-01-15|
| Meena Das|  Health|  1200|      2025-01-23|
+----------+--------+------+----------------+



## Users With Potential Spending Spike

In [7]:
from pyspark.sql.functions import round as spark_round

user_summary = df.groupBy("user_id", "user_name")     .agg(
        sum("amount").alias("total_spend"),
        avg("amount").alias("avg_per_txn"),
        count("amount").alias("txn_count")
    )     .withColumn("avg_per_txn", spark_round(col("avg_per_txn"), 2))

spike_threshold = avg_amount * 1.5
flagged = user_summary.filter(col("avg_per_txn") > spike_threshold)

print(f"Spike threshold (1.5x average): ₹{spike_threshold:.2f}")
print("Users with above-average spending patterns:")
flagged.show()

Spike threshold (1.5x average): ₹278.04
Users with above-average spending patterns:
+-------+---------+-----------+-----------+---------+
|user_id|user_name|total_spend|avg_per_txn|txn_count|
+-------+---------+-----------+-----------+---------+
+-------+---------+-----------+-----------+---------+



## Save Output

In [8]:
monthly_spend.coalesce(1).write.csv("pyspark_monthly_output", header=True, mode="overwrite")
print("Output saved to pyspark_monthly_output")

Output saved to pyspark_monthly_output/
